In [3]:
import json
import os
import urllib.request
from pathlib import Path

url = 'https://www.krl.dk/sirka/sirkaApi/tableApi'
api_key = os.environ.get(
    'SIRKA_API_KEY',
    'b6caa39c5a6dc520a83ed1624a7bdfdd1935d9fb2bee1d5e2c14dc08ee4ea57ceacca8e9550f5b25ad07b745a612d4b6b5b07bd781baff0908b5891c0beb65c1',
)

def find_excel_dir(start: Path) -> Path:
    start = start.resolve()
    for p in (start, *start.parents):
        if (p / 'EXCEL').exists():
            return p / 'EXCEL'
        if (p / 'Excel').exists():
            return p / 'Excel'
    return start

excel_dir = find_excel_dir(Path.cwd())
out_dir = excel_dir / 'Data' / 'figur3'
out_dir.mkdir(parents=True, exist_ok=True)

def post_json_excel(*, url: str, payload: dict, timeout: int = 180) -> bytes:
    body = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(
        url,
        data=body,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()

payload = {
    'apiKey': api_key,
    'table': 'Personale-måned',
    'time': [
        {'y1': '2025', 'm1': '02'},
        {'y1': '2024', 'm1': '02'},
        {'y1': '2023', 'm1': '02'},
        {'y1': '2022', 'm1': '02'},
        {'y1': '2021', 'm1': '02'},
    ],
    'control': ['overenskomst'],
    'data': ['fuldtid'],
    'selection': [
        {
            'name': 'Udvalgte population',
            'filters': {
                'trs': ['272', '285', '293', '072'],
                'omr': ['1', '8'],
            },
        }
    ],
    'options': {
        'totals': True,
        'outputFormat': 'excel',
        'actions': [],
        'tableName': 'Antal ansatte',
        'subLimit': 5,
        'modelName': 'SIRKA',
        'timeIncreasing': True,
    },
    'dimension': {
        'viewportHeight': 812,
        'viewportWidth': 1440,
        'xsMaxWidth': 768,
        'smMaxWidth': 992,
        'mdMaxWidth': 1200,
        'CONSTANTS': {'XS': 0, 'SM': 1, 'MD': 2, 'LG': 3, 'MAIL': 4},
    },
}

content = post_json_excel(url=url, payload=payload, timeout=180)
out_path = out_dir / 'figur3overenskomst.xlsx'
out_path.write_bytes(content)
print('Wrote:', out_path.resolve())

Wrote: /Users/magnusrasmussen/Desktop/Arbejde/EXCEL/Data/figur3/figur3overenskomst.xlsx
